In [69]:

import numpy as np
import pandas as pd
import joblib

from scipy import stats

from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import OneHotEncoder,StandardScaler,FunctionTransformer

from sklearn.pipeline import Pipeline, make_pipeline

from sklearn.compose import ColumnTransformer

from sklearn.metrics.pairwise import rbf_kernel

from sklearn.tree import DecisionTreeRegressor

from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score,root_mean_squared_error


# Load Database

In [70]:

housing = pd.read_csv(
    "california_housing_prices.csv"
)

# Stratified Train / Test Split

In [71]:

housing["income_category"] = pd.cut(
    housing["median_income"],
    bins=[0., 1.5, 3., 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

train_set, test_set = train_test_split(
    housing,
    test_size=0.2,
    random_state=40,
    stratify=housing["income_category"]
)

train_set = train_set.drop(columns="income_category")
test_set = test_set.drop(columns="income_category")


# Separate Features & Target


In [72]:

train_features = train_set.drop(
    columns="median_house_value"
)

train_target = train_set["median_house_value"]

test_features = test_set.drop(
    columns="median_house_value"
)

test_target = test_set["median_house_value"]


# Feature Engineering Functions

In [73]:

def ratio_of_columns(X):
    return X[:, [0]] / X[:, [1]]


def ratio_feature_name(transformer, feature_names_in):
    return ["ratio"]



# Feature Engineering Pipelines

In [74]:

def ratio_pipeline():

    return make_pipeline(

        SimpleImputer(strategy="median"),

        FunctionTransformer(
            ratio_of_columns,
            feature_names_out=ratio_feature_name
        ),

        StandardScaler()

    )


heavy_tail_pipeline = make_pipeline(

    SimpleImputer(strategy="median"),

    FunctionTransformer(
        np.log,
        feature_names_out="one-to-one"
    ),

    StandardScaler()

)

housing_age_pipeline = FunctionTransformer(

    rbf_kernel,

    feature_names_out="one-to-one",

    kw_args={
        "Y": [[35]],
        "gamma": 0.1
    }

)

default_numeric_pipeline = make_pipeline(

    SimpleImputer(strategy="median"),

    StandardScaler()

)

categorical_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),

    (
        "encoder",
        OneHotEncoder()
    )

])

In [75]:

all_transformations = ColumnTransformer(

    [

        (
            "bedrooms_per_room",
            ratio_pipeline(),
            ["total_bedrooms", "total_rooms"]
        ),

        (
            "rooms_per_house",
            ratio_pipeline(),
            ["total_rooms", "households"]
        ),

        (
            "people_per_house",
            ratio_pipeline(),
            ["population", "households"]
        ),

        (
            "log_features",
            heavy_tail_pipeline,
            [
                "total_bedrooms",
                "total_rooms",
                "population",
                "households",
                "median_income"
            ]
        ),

        (
            "housing_age_rbf",
            housing_age_pipeline,
            ["housing_median_age"]
        ),

        (
            "categorical",
            categorical_pipeline,
            ["ocean_proximity"]
        )

    ],

    remainder=default_numeric_pipeline

)


# Transform Training Data

In [76]:

train_features_transformed = all_transformations.fit_transform(
    train_features
)


# Feature Names

In [77]:
feature_names = all_transformations.get_feature_names_out()

# Linear Regression Pipeline

In [78]:

linear_regression_pipeline = Pipeline(

    [

        (
            "preprocessing",
            all_transformations
        ),

        (
            "model",
            LinearRegression()
        )

    ]

)

# Decision Tree Pipeline

In [79]:

decision_tree_pipeline = Pipeline(

    [

        (
            "preprocessing",
            all_transformations
        ),

        (
            "model",
            DecisionTreeRegressor(
                random_state=40
            )
        )

    ]

)

# Train Linear Regression

In [80]:

linear_regression_pipeline.fit(

    train_features,
    train_target

)

linear_predictions = linear_regression_pipeline.predict(
    train_features
)


# Linear Regression Metrics

In [81]:


linear_mae = mean_absolute_error(
    train_target,
    linear_predictions
)

linear_rmse = mean_squared_error(
    train_target,
    linear_predictions,
   )

linear_root = root_mean_squared_error(
    train_target,
    linear_predictions,
   )

linear_r2 = r2_score(
    train_target,
    linear_predictions
)

print("=" * 60)
print("Linear Regression")
print("=" * 60)

print(f"MAE  : {linear_mae:.2f}")
print(f"RMSE : {linear_rmse:.2f}")
print(f"root : {linear_root:.2f}")
print(f"R²   : {linear_r2:.4f}")


Linear Regression
MAE  : 53721.06
RMSE : 5169319861.14
root : 71897.98
R²   : 0.6097


# Cross Validation

In [82]:

linear_cv = -cross_val_score(

    linear_regression_pipeline,

    train_features,

    train_target,

    scoring="neg_mean_absolute_error",

    cv=5

)

print("\nCross Validation (Linear Regression)")
print(pd.Series(linear_cv).describe())



Cross Validation (Linear Regression)
count        5.000000
mean     53858.315912
std        745.455015
min      52921.895396
25%      53293.868044
50%      54107.355312
75%      54182.759503
max      54785.701308
dtype: float64


# Train Decision Tree

In [83]:

decision_tree_pipeline.fit(

    train_features,
    train_target

)

decision_tree_predictions = decision_tree_pipeline.predict(
    train_features
)

# Decision Tree Metrics


In [84]:

decision_tree_mae = mean_absolute_error(
    train_target,
    decision_tree_predictions
)

decision_tree_rmse = mean_squared_error(
    train_target,
    decision_tree_predictions
)

decision_tree_root = root_mean_squared_error(
    train_target,
    decision_tree_predictions
)

decision_tree_r2 = r2_score(
    train_target,
    decision_tree_predictions
)

print("\n" + "=" * 60)
print("Decision Tree")
print("=" * 60)

print(f"MAE  : {decision_tree_mae:.2f}")
print(f"RMSE : {decision_tree_rmse:.2f}")
print(f"root : {decision_tree_root:.2f}")
print(f"R²   : {decision_tree_r2:.4f}")



Decision Tree
MAE  : 0.00
RMSE : 0.00
root : 0.00
R²   : 1.0000


# Cross Validation (Decision Tree)

In [85]:

decision_tree_cv = -cross_val_score(

    decision_tree_pipeline,

    train_features,

    train_target,

    scoring="neg_mean_absolute_error",

    cv=5

)

print("\nCross Validation (Decision Tree)")
print(pd.Series(decision_tree_cv).describe())



Cross Validation (Decision Tree)
count        5.000000
mean     45860.774270
std       1048.685863
min      44529.378444
25%      45000.431426
50%      46307.648395
75%      46446.683525
max      47019.729558
dtype: float64


# Compare Models

In [86]:

comparison = pd.DataFrame({

    "Model": [

        "Linear Regression",
        "Decision Tree"

    ],

    "MAE": [

        linear_mae,
        decision_tree_mae

    ],

    "RMSE": [

        linear_rmse,
        decision_tree_rmse

    ],

    "ROOT": [

        linear_root,
        decision_tree_root

    ],


    "R2": [

        linear_r2,
        decision_tree_r2

    ],

    "Cross Validation MAE": [

        linear_cv.mean(),
        decision_tree_cv.mean()

    ]

})

print("\n")
print("=" * 60)
print("Model Comparison")
print("=" * 60)
print(comparison)




Model Comparison
               Model           MAE          RMSE          ROOT        R2  \
0  Linear Regression  53721.057076  5.169320e+09  71897.982316  0.609697   
1      Decision Tree      0.000000  0.000000e+00      0.000000  1.000000   

   Cross Validation MAE  
0          53858.315912  
1          45860.774270  


# Hyperparameter Tuning

In [87]:

grid_parameters = {

    "preprocessing__housing_age_rbf__kw_args": [

        {

            "Y": [[35.0]],
            "gamma": 0.1

        },

        {

            "Y": [[35.0]],
            "gamma": 0.2

        }

    ],

    "model__max_depth": [

        5,
        10,
        15

    ]

}

grid_search = GridSearchCV(

    estimator=decision_tree_pipeline,

    param_grid=grid_parameters,

    scoring="neg_mean_absolute_error",

    cv=5,

    n_jobs=1

)

grid_search.fit(

    train_features,
    train_target

)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=40))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [5, 10, ...], 'preprocessing__housing_age_rbf__kw_args': [{'Y': [[35.0]], 'gamma': 0.1}, {'Y': [[35.0]], 'gamma': 0.2}]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Supp

# Best Model

In [88]:

best_model = grid_search.best_estimator_

print("\n")
print("=" * 60)
print("Best Parameters")
print("=" * 60)

print(grid_search.best_params_)




Best Parameters
{'model__max_depth': 10, 'preprocessing__housing_age_rbf__kw_args': {'Y': [[35.0]], 'gamma': 0.2}}


# Cross Validation Results

In [89]:

cv_results = pd.DataFrame(

    grid_search.cv_results_

)

cv_results = cv_results.sort_values(

    by="mean_test_score",

    ascending=False

)

print("\nTop GridSearch Results")

print(

    cv_results[

        [

            "params",
            "mean_test_score",
            "std_test_score"

        ]

    ].head(10)

)



Top GridSearch Results
                                              params  mean_test_score  \
3  {'model__max_depth': 10, 'preprocessing__housi...    -41331.364039   
2  {'model__max_depth': 10, 'preprocessing__housi...    -41575.473114   
5  {'model__max_depth': 15, 'preprocessing__housi...    -43525.331672   
4  {'model__max_depth': 15, 'preprocessing__housi...    -43699.893801   
1  {'model__max_depth': 5, 'preprocessing__housin...    -48733.165935   
0  {'model__max_depth': 5, 'preprocessing__housin...    -48733.165935   

   std_test_score  
3      538.723628  
2      379.882404  
5      933.292571  
4      840.463363  
1      553.161808  
0      553.161808  


# Evaluate Best Model on Test Set

In [90]:

test_predictions = best_model.predict(test_features)

test_mae = mean_absolute_error(
    test_target,
    test_predictions
)

test_rmse = mean_squared_error(
    test_target,
    test_predictions
)

test_root = root_mean_squared_error(
    test_target,
    test_predictions
)

test_r2 = r2_score(
    test_target,
    test_predictions
)

print("\n")
print("=" * 60)
print("Final Test Performance")
print("=" * 60)

print(f"MAE  : {test_mae:.2f}")
print(f"RMSE : {test_rmse:.2f}")
print(f"ROOT : {test_root:.2f}")
print(f"R²   : {test_r2:.4f}")



Final Test Performance
MAE  : 40610.18
RMSE : 3807631103.77
ROOT : 61706.01
R²   : 0.7200


# Feature Importance

In [91]:

feature_importance = pd.DataFrame({

    "Feature": feature_names,

    "Importance": best_model.named_steps[
        "model"
    ].feature_importances_

})

feature_importance = feature_importance.sort_values(

    by="Importance",

    ascending=False

)

print("\n")
print("=" * 60)
print("Top 20 Most Important Features")
print("=" * 60)

print(feature_importance.head(20))




Top 20 Most Important Features
                                    Feature  Importance
7               log_features__median_income    0.541188
10      categorical__ocean_proximity_INLAND    0.168910
2                   people_per_house__ratio    0.116458
14                     remainder__longitude    0.065758
15                      remainder__latitude    0.048133
1                    rooms_per_house__ratio    0.016768
0                  bedrooms_per_room__ratio    0.013966
8       housing_age_rbf__housing_median_age    0.010388
6                  log_features__households    0.005713
4                 log_features__total_rooms    0.004695
3              log_features__total_bedrooms    0.004501
5                  log_features__population    0.002806
13  categorical__ocean_proximity_NEAR OCEAN    0.000386
9    categorical__ocean_proximity_<1H OCEAN    0.000167
12    categorical__ocean_proximity_NEAR BAY    0.000163
11      categorical__ocean_proximity_ISLAND    0.000000


# Confidence Interval of RMSE

In [92]:

squared_errors = (

    test_predictions -
    test_target

) ** 2

confidence = 0.95

rmse_interval = np.sqrt(

    stats.t.interval(

        confidence,

        len(squared_errors) - 1,

        loc=squared_errors.mean(),

        scale=stats.sem(squared_errors)

    )

)

print("\n")
print("=" * 60)
print("95% Confidence Interval (RMSE)")
print("=" * 60)

print(rmse_interval)




95% Confidence Interval (RMSE)
[59022.26026965 64277.79554561]


# Save Best Model

In [93]:

joblib.dump(

    best_model,

    "housing_model.pkl"

)

['housing_model.pkl']

# Load Model

In [94]:

loaded_model = joblib.load(
    "housing_model.pkl"
)


# Predict First Five Houses

In [95]:

sample_predictions = loaded_model.predict(

    test_features.iloc[:5]

)

prediction_table = pd.DataFrame({

    "Actual Price": test_target.iloc[:5].values,

    "Predicted Price": np.round(
        sample_predictions,
        0
    )

})

print("\n")
print("=" * 60)
print("Sample Predictions")
print("=" * 60)

print(prediction_table)




Sample Predictions
   Actual Price  Predicted Price
0      160300.0         156769.0
1      150000.0         121957.0
2      311300.0         220713.0
3      149000.0         121957.0
4      112500.0          92462.0
